In [12]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

spark = SparkSession.builder \
    .appName("Pertemuan13-ModelPersistence") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [13]:
from pymongo import MongoClient
import random

client = MongoClient("mongodb://localhost:27017/")
db_ecom = client["ecommerce_p13"]
koleksi_transaksi = db_ecom["transaksi_mentah"]
koleksi_transaksi.delete_many({})  # membersihkan data lama agar percobaan konsisten

random.seed(99)
n_transaksi = 900
kategori_ecom = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
kota_ecom = ["Magelang", "Yogyakarta", "Semarang", "Solo"]
metode_ecom = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

dokumen_transaksi = []
for _ in range(n_transaksi):
    estimasi_hari = random.randint(1, 14)
    # Semakin lama estimasi_pengiriman_hari, semakin besar probabilitas dibatalkan (label = 1)
    prob_batal = 1 / (1 + (2.718281828459045 ** (-(estimasi_hari - 7) / 2.5)))
    dibatalkan = 1 if random.random() < prob_batal else 0
    dokumen_transaksi.append({
        "kategori": random.choice(kategori_ecom),
        "kota": random.choice(kota_ecom),
        "unit_terjual": random.randint(1, 10),
        "harga_satuan": random.choice([25000, 50000, 100000, 250000, 500000]),
        "metode_pembayaran": random.choice(metode_ecom),
        "estimasi_pengiriman_hari": estimasi_hari,
        "dibatalkan": dibatalkan,
    })

koleksi_transaksi.insert_many(dokumen_transaksi)
print(f"Total dokumen tersisip di ecommerce_p13.transaksi_mentah: {koleksi_transaksi.count_documents({})}")

Total dokumen tersisip di ecommerce_p13.transaksi_mentah: 900


In [14]:
import pandas as pd

data_mongo_ecom = list(koleksi_transaksi.find({}, {"_id": 0}))
df_ecom_pd = pd.DataFrame(data_mongo_ecom)
df_ecom = spark.createDataFrame(df_ecom_pd)

df_ecom.printSchema()
df_ecom.show(5)
print("Jumlah baris Spark DataFrame:", df_ecom.count())

root
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: long (nullable = true)
 |-- harga_satuan: long (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- estimasi_pengiriman_hari: long (nullable = true)
 |-- dibatalkan: long (nullable = true)

+------------+----------+------------+------------+-----------------+------------------------+----------+
|    kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|estimasi_pengiriman_hari|dibatalkan|
+------------+----------+------------+------------+-----------------+------------------------+----------+
|   Kesehatan|Yogyakarta|           4|       50000|         E-Wallet|                       7|         1|
|Rumah Tangga|  Magelang|          10|      250000|         E-Wallet|                      13|         1|
|   Kesehatan|Yogyakarta|           6|      250000|         E-Wallet|                       7|         0|
|     Fashion|  Semarang|           2|      500000|    

In [16]:
from pyspark.ml.feature import OneHotEncoder
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

idx_kat = StringIndexer(inputCol="kategori", outputCol="kategori_idx")
idx_kota = StringIndexer(inputCol="kota", outputCol="kota_idx")
idx_metode = StringIndexer(inputCol="metode_pembayaran", outputCol="metode_idx")

ohe_ecom = OneHotEncoder(
    inputCols=["kategori_idx", "kota_idx", "metode_idx"],
    outputCols=["kategori_vec", "kota_vec", "metode_vec"]
)

assembler_ecom = VectorAssembler(
    inputCols=["kategori_vec", "kota_vec", "metode_vec", "unit_terjual", "harga_satuan", "estimasi_pengiriman_hari"],
    outputCol="features"
)

rf_ecom = RandomForestClassifier(featuresCol="features", labelCol="dibatalkan", numTrees=30, seed=99)

pipeline_ecom = Pipeline(stages=[idx_kat, idx_kota, idx_metode, ohe_ecom, assembler_ecom, rf_ecom])

train_e, test_e = df_ecom.randomSplit([0.8, 0.2], seed=99)
model_ecom = pipeline_ecom.fit(train_e)
pred_e = model_ecom.transform(test_e)

ev_acc_e = MulticlassClassificationEvaluator(labelCol="dibatalkan", predictionCol="prediction", metricName="accuracy")
ev_auc_e = BinaryClassificationEvaluator(labelCol="dibatalkan", metricName="areaUnderROC")

acc_ecom = ev_acc_e.evaluate(pred_e)
auc_ecom = ev_auc_e.evaluate(pred_e)

print(f"Accuracy : {acc_ecom:.4f}")
print(f"AUC-ROC  : {auc_ecom:.4f}")

path_model_ecom = "model_pembatalan_tersimpan"
model_ecom.write().overwrite().save(path_model_ecom)
print(f"Model berhasil disimpan ke folder '{path_model_ecom}'")

Accuracy : 0.7380
AUC-ROC  : 0.7695


Model berhasil disimpan ke folder 'model_pembatalan_tersimpan'


In [17]:
spark.stop()
print("SparkSession lama dihentikan untuk simulasi minggu berikutnya.")

SparkSession lama dihentikan untuk simulasi minggu berikutnya.


In [19]:
from pyspark.ml import PipelineModel

spark = SparkSession.builder \
    .appName("Pertemuan13-MingguBerikutnya") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

model_minggu_depan = PipelineModel.load("model_pembatalan_tersimpan")

transaksi_baru_10 = spark.createDataFrame(pd.DataFrame({
    "kategori": ["Elektronik", "Fashion", "Makanan", "Kesehatan", "Rumah Tangga", "Elektronik", "Fashion", "Kesehatan", "Makanan", "Rumah Tangga"],
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Magelang", "Yogyakarta", "Semarang", "Solo", "Magelang", "Yogyakarta"],
    "metode_pembayaran": ["COD", "E-Wallet", "Transfer Bank", "Kartu Kredit", "COD", "Transfer Bank", "E-Wallet", "COD", "Kartu Kredit", "E-Wallet"],
    "unit_terjual": [1, 3, 2, 1, 4, 2, 5, 1, 3, 2],
    "harga_satuan": [500000, 100000, 25000, 250000, 50000, 250000, 100000, 100000, 50000, 250000],
    "estimasi_pengiriman_hari": [12, 3, 2, 10, 13, 4, 6, 11, 2, 14],
}))

hasil_prediksi_baru = model_minggu_depan.transform(transaksi_baru_10)
hasil_prediksi_baru.select(
    "kategori", "kota", "metode_pembayaran", "estimasi_pengiriman_hari", "prediction", "probability"
).show(truncate=False)

+------------+----------+-----------------+------------------------+----------+----------------------------------------+
|kategori    |kota      |metode_pembayaran|estimasi_pengiriman_hari|prediction|probability                             |
+------------+----------+-----------------+------------------------+----------+----------------------------------------+
|Elektronik  |Magelang  |COD              |12                      |1.0       |[0.20093395495195532,0.7990660450480447]|
|Fashion     |Yogyakarta|E-Wallet         |3                       |0.0       |[0.8635660167684885,0.13643398323151154]|
|Makanan     |Semarang  |Transfer Bank    |2                       |0.0       |[0.819979193479287,0.18002080652071303] |
|Kesehatan   |Solo      |Kartu Kredit     |10                      |1.0       |[0.2571019074920888,0.7428980925079112] |
|Rumah Tangga|Magelang  |COD              |13                      |1.0       |[0.15698205949893526,0.8430179405010648]|
|Elektronik  |Yogyakarta|Transfe

In [20]:
output_laporan_ecom = "laporan_risiko_pembatalan"
hasil_prediksi_baru.write.mode("overwrite").partitionBy("kategori").parquet(output_laporan_ecom)

df_verifikasi_ecom = spark.read.parquet(output_laporan_ecom)
print("Jumlah baris hasil verifikasi laporan Parquet:", df_verifikasi_ecom.count())
!ls -la {output_laporan_ecom}

Jumlah baris hasil verifikasi laporan Parquet: 10
total 32
drwxr-xr-x  7 yue yue 4096 Nov 26 14:51  .
drwxrwxr-x 22 yue yue 4096 Nov 26 14:50  ..
drwxr-xr-x  2 yue yue 4096 Nov 26 14:51 'kategori=Elektronik'
drwxr-xr-x  2 yue yue 4096 Nov 26 14:51 'kategori=Fashion'
drwxr-xr-x  2 yue yue 4096 Nov 26 14:51 'kategori=Kesehatan'
drwxr-xr-x  2 yue yue 4096 Nov 26 14:51 'kategori=Makanan'
drwxr-xr-x  2 yue yue 4096 Nov 26 14:51 'kategori=Rumah Tangga'
-rw-r--r--  1 yue yue    0 Nov 26 14:51  _SUCCESS
-rw-r--r--  1 yue yue    8 Nov 26 14:51  ._SUCCESS.crc


Alur kerja analitik end-to-end yang telah dibangun dimulai dari penyimpanan data mentah transaksi e-commerce ke dalam basis data NoSQL MongoDB (ecommerce_p13.transaksi_mentah) dengan memanfaatkan skema fleksibel berbasis dokumen. Selanjutnya, data diekstraksi dan dikonversi menjadi Spark DataFrame untuk diproses secara terdistribusi (distributed processing) menggunakan Apache Spark melalui rangkaian Pipeline (StringIndexer, OneHotEncoder, VectorAssembler, dan RandomForestClassifier) guna memprediksi risiko pembatalan transaksi (dibatalkan). Model yang telah dilatih disimpan secara persisten melalui mekanisme model persistence (model_pembatalan_tersimpan), sehingga pada simulasi minggu berikutnya dapat dimuat ulang pada SparkSession yang baru menggunakan PipelineModel.load() untuk melakukan inferensi data baru tanpa perlu melatih ulang model dari awal. Hasil prediksi kemudian disimpan ke format Apache Parquet berbasis columnar storage dengan partisi per kategori (laporan_risiko_pembatalan) serta diverifikasi ulang jumlah barisnya. Berdasarkan pola data dan hasil prediksi risiko pembatalan yang dipengaruhi oleh estimasi_pengiriman_hari, rekomendasi bisnis konkret bagi tim operasional adalah menetapkan batas SLA estimasi pengiriman yang lebih ketat pada rute atau wilayah dengan potensi keterlambatan tinggi serta memberikan notifikasi status pengiriman proaktif atau opsi penyesuaian layanan pengiriman pada pesanan dengan estimasi hari yang panjang agar pelanggan tidak membatalkan transaksi secara sepihak.

In [21]:
client.close()
spark.stop()
print("Koneksi MongoDB & SparkSession ditutup.")

Koneksi MongoDB & SparkSession ditutup.
